# 02f — the post-hoc arm and the baselines

**Everything in this notebook was decided on 2026-08-23, *after* the `prereg-v2` tag.**
None of it is registered, and the paper must label all of it post-hoc where it is reported.
It is kept in its own notebook, writing to its own output directories, so that a later
reader cannot mistake it for part of the 36-run registered sweep.

Estimated **~2 h** on a T4. Suggested kernel name: `emocap-posthoc`.

## What it does, and why each part exists

**Part A — `S_paired_matched`, 6 runs (~20 min).** As registered, P3 compares `S_paired5`
against `S_unpaired` and moves three things at once: 9× the cells, roughly twice the
images, **and** the paired structure. The prereg says so plainly — it calls P3a "a sanity
check on the extra data". This arm holds volume fixed: 878 images × 5 registers = **4,390
cells**, exactly `S_unpaired`'s size, differing only in structure. If it clears its keyword
anchor where `S_unpaired` does not, pairing is isolated from volume, and the claim stops
being "more data helps".

Its cells are a strict subset of `S_paired5`'s and its images a strict subset of
`S_unpaired`'s — `scripts/build_posthoc_arms.py` asserts all three containments before it
will write the file. Nothing new was sampled, so "the post-hoc arm drew easier data" is not
available as an explanation of whatever it shows.

**Part B — baselines (~1.5 h).** Every arm is measured against its keyword anchor, which
says what a lexical rule reaches with no model. Nothing yet says what GPT-2 reaches with no
**image**. Two modes, both frozen weights, no training:

| mode | prompt | what it measures |
|---|---|---|
| `prior` | the register word alone | the pure language prior. Deterministic, so it collapses to **5 captions total** — a floor, not a measurement |
| `text`  | register word **+** the neutral Flickr8k source caption | how much accuracy is reachable from the source text with the photograph removed |

`prior` is the baseline named in `docs/remaining-work.md`. Its collapse is the same trap the
blanked visual-dependence probe walked into, where a negative control appeared to score
0.58 over five outcomes; the script prints the five captions so it cannot be read as an
effect. `text` is the one that carries information.

`S_paired25` gets `prior` but **not** `text`: 40,235 cells × 5 folds is ~4 GPU hours to
measure a frozen model that never sees the image, and its cells sit on the same images as
`S_paired5`'s.

**Report margins, not raw accuracy, against these.** The study's central finding is that
most of the primary metric is keyword-reachable; leaning on raw accuracy to praise a model
would contradict its own argument.

## Setup

**Attach:** dataset `emocap-v2-arms` — it must be a push made *after* `S_paired_matched.jsonl`
existed, or Part A fails on a missing arm file. **Accelerator:** GPU **T4 x2**.
**Internet:** on.

> P100 will not work. It is compute capability sm_60 and Kaggle's PyTorch build ships
> kernels for sm_70 and up, so every launch fails with `no kernel image is available`.

Both parts are resumable: a re-run skips whatever already landed, so a dead session costs
only the runs that had not completed.

Pull the results:

    kaggle kernels output <owner>/emocap-posthoc -p tmp/emocap-posthoc --page-size 200 \
        --file-pattern 'runs/.*\.(jsonl|json)$'


In [ ]:
# ── parameters ────────────────────────────────────────────────────────────
# Part A: (ARM, FOLD, NEGATIVE_CONTROL), executed in order.
TRAIN_RUNS = [
    ("S_paired_matched", 0, False),
    ("S_paired_matched", 1, False),
    ("S_paired_matched", 2, False),
    ("S_paired_matched", 3, False),
    ("S_paired_matched", 4, False),
    ("S_paired_matched", 0, True),
]

# Part B: (MODE, CELLS_FROM_ARM, FOLD). `prior` is ~5 generations per entry and free;
# `text` costs one beam search per held-out cell, which is why S_paired25 is prior-only.
BASELINES = (
    [("prior", arm, f)
     for arm in ("S_paired5", "S_unpaired", "S_paired_matched", "S_paired25")
     for f in range(5)]
    + [("text", arm, f)
       for arm in ("S_paired5", "S_unpaired", "S_paired_matched")
       for f in range(5)]
)

SEED = 42

DATA = "/kaggle/input/emocap-v2-arms"
OUT = "/kaggle/working/runs"              # Part A, matching the registered sweep's layout
OUT_BASE = "/kaggle/working/runs/baselines"  # Part B, kept separate on purpose


In [ ]:
# Clone the EXACT commit the data was built from. Pinning to the commit recorded in
# provenance.json is what stops a notebook from pairing this dataset version with a
# different version of the code -- a mismatch that would be silent and unrecoverable.
import json, os, subprocess
from pathlib import Path

COMMIT = json.loads(Path(DATA, "provenance.json").read_text())["git_commit"]
if not Path("/kaggle/working/EmoCap").exists():
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/asjad2401/EmoCap.git",
                    "/kaggle/working/EmoCap"], check=True)
subprocess.run(["git", "-C", "/kaggle/working/EmoCap", "checkout", "-q", COMMIT], check=True)
print("code at", COMMIT[:12])


In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/EmoCap/src")
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Fail here rather than 20 minutes in. The post-hoc arm did not exist when the registered
# arms were pushed, so an older dataset version has every other arm and not this one --
# a missing-file error mid-batch that names the wrong problem.
arm_file = Path(DATA, "arms", "S_paired_matched.jsonl")
if not arm_file.exists():
    raise SystemExit(
        "S_paired_matched.jsonl is not in the attached dataset.\n"
        "Run `uv run python scripts/build_posthoc_arms.py` locally, then\n"
        "`uv run python scripts/push_kaggle.py`, then attach the new version.")
print(f"{arm_file.name}: {sum(1 for _ in arm_file.open()):,} cells")

man = Path(DATA, "arms", "posthoc_manifest.json")
if man.exists():
    print(json.dumps(json.loads(man.read_text())["nested_in"], indent=2))


## Part A — `S_paired_matched`

Six runs, identical hyperparameters to every registered run. `train_arm.py` takes the arm
name as a data-file selector and nothing else, so this arm is trained by exactly the same
code path as the six registered ones — which is the property that makes it comparable.

In [ ]:
import time

results = []
t_all = time.time()

for i, (arm, fold, nc) in enumerate(TRAIN_RUNS, 1):
    tag = f"{arm}-f{fold}" + ("-nc" if nc else "")
    run_dir = Path(OUT, tag)

    if (run_dir / "predictions.jsonl").exists():
        print(f"[A {i}/{len(TRAIN_RUNS)}] {tag}: already present, skipping\n", flush=True)
        results.append((tag, "skipped", 0.0))
        continue

    cmd = [sys.executable, "-u", "/kaggle/working/EmoCap/scripts/train_arm.py",
           "--arm", arm, "--fold", str(fold), "--seed", str(SEED),
           "--data-root", DATA, "--out-root", OUT]
    if nc:
        cmd.append("--negative-control")

    print(f"[A {i}/{len(TRAIN_RUNS)}] {tag}   ({(time.time()-t_all)/60:.1f} min into batch)",
          flush=True)
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print("   ", line, end="")
    rc = proc.wait()

    # A failure does NOT abort the batch. The runs are independent, and losing five good
    # ones because the fourth crashed is exactly the failure this loop exists to avoid.
    status = "ok" if rc == 0 else f"FAILED rc={rc}"
    results.append((tag, status, round((time.time() - t0) / 60, 1)))
    print(f"    -> {status}  [{results[-1][2]} min]\n", flush=True)


## Part B — the baselines

No training happens here: `baseline_prior.py` loads the same frozen GPT-2 the arms use and
decodes with the same frozen `DecodeConfig`. `prior` entries finish in seconds because the
prompt depends only on the register, so five beam searches are the entire output.

In [ ]:
base_results = []
t_all = time.time()

for i, (mode, arm, fold) in enumerate(BASELINES, 1):
    tag = f"BASE_{mode}-{arm}-f{fold}"
    run_dir = Path(OUT_BASE, tag)

    if (run_dir / "predictions.jsonl").exists():
        print(f"[B {i}/{len(BASELINES)}] {tag}: already present, skipping", flush=True)
        base_results.append((tag, "skipped", 0.0))
        continue

    cmd = [sys.executable, "-u", "/kaggle/working/EmoCap/scripts/baseline_prior.py",
           "--arm", arm, "--fold", str(fold), "--mode", mode, "--seed", str(SEED),
           "--data-root", DATA, "--out-root", OUT_BASE]

    print(f"[B {i}/{len(BASELINES)}] {tag}   ({(time.time()-t_all)/60:.1f} min into batch)",
          flush=True)
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print("   ", line, end="")
    rc = proc.wait()
    status = "ok" if rc == 0 else f"FAILED rc={rc}"
    base_results.append((tag, status, round((time.time() - t0) / 60, 1)))
    print(f"    -> {status}  [{base_results[-1][2]} min]\n", flush=True)


In [ ]:
# What landed. `empty_captions` above zero means decode collapsed and that run is suspect;
# for a `prior` baseline, `unique_captions` of 5 is EXPECTED and is the point.
print(f"{'run':<34} {'status':<14} {'min':>6}")
for tag, status, mins in results + base_results:
    print(f"{tag:<34} {status:<14} {mins:>6.1f}")

print("\n── Part A ──")
for tag, status, _ in results:
    if status != "ok":
        continue
    st = json.loads(Path(OUT, tag, "stats.json").read_text())
    n = sum(1 for _ in Path(OUT, tag, "predictions.jsonl").open())
    print(f"{tag}: {n:,} decoded, {st['empty_captions']} empty, "
          f"loss {st['losses'][0]:.3f} -> {st['losses'][-1]:.3f}, "
          f"probe {st['novis_identical_rate']:.1%} unchanged, {st['wall_minutes']} min")

print("\n── Part B ──")
for tag, status, _ in base_results:
    if status != "ok":
        continue
    st = json.loads(Path(OUT_BASE, tag, "stats.json").read_text())
    print(f"{tag}: {st['decoded']:,} cells, {st['unique_captions']} unique captions, "
          f"{st['wall_minutes']} min")

# Print the prior's five captions once. They are the whole content of that baseline and
# belong in the paper beside its accuracy, so that nobody reads a five-outcome draw as an
# effect.
for tag, status, _ in base_results:
    if status == "ok" and tag.startswith("BASE_prior-") and tag.endswith("-f0"):
        st = json.loads(Path(OUT_BASE, tag, "stats.json").read_text())
        print(f"\n{tag} -- the language prior's entire output:")
        for reg, cap in st.get("captions_by_register", {}).items():
            print(f"  {reg:<9} {cap!r}")
        break

failed = [t for t, s, _ in results + base_results if s.startswith("FAILED")]
if failed:
    raise RuntimeError(f"{len(failed)} run(s) failed: {failed}")
